# Global PCA for Industrial Anomaly Detection

**CS 663 — Digital Image Processing | MVTec AD**

This notebook implements Principal Component Analysis from the underlying linear algebra. The implementation does not use a ready-made PCA class.

Each complete grayscale image is flattened into one feature vector. PCA learns the dominant patterns from normal training images. An image is projected onto the learned principal components and reconstructed. Pixels with a large reconstruction error are treated as possible defects.

The training split contains only normal images. Validation images are used to set the image-level threshold. The official test split is reserved for final evaluation.

The global image covariance matrix would be extremely large at 256 × 256 pixels. We therefore use the mathematically equivalent sample-space formulation, which eigendecomposes a matrix whose size is the number of training images.

## 1. Setup and data loading

Run this notebook from the repository in Colab or from a local clone. The default category is bottle. Change CATEGORY to evaluate another category.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from scipy.ndimage import label as connected_components

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO = Path("/content/CS-663")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/pranitas-cloud/CS-663.git", str(REPO)], check=True)
else:
    REPO = Path.cwd()
    if not (REPO / "src").is_dir():
        raise RuntimeError("Open this notebook from the CS-663 repository root.")

os.chdir(REPO)
if str(REPO.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO.resolve()))

from src.data.pipelines import load_config, build_dataset, stack_images

CONFIG = load_config("configs/default.yaml")
DATA_ROOT = Path(os.environ.get("MVTEC_DATA_ROOT", REPO / "data" / "mvtec_anomaly_detection")).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = REPO / DATA_ROOT
CONFIG["data"]["root"] = str(DATA_ROOT)

CATEGORY = "bottle"
IMAGE_SIZE = int(CONFIG["pca"].get("image_size", 256))
SEED = int(CONFIG.get("seed", 42))
np.random.seed(SEED)
torch.manual_seed(SEED)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"MVTec folder not found: {DATA_ROOT}")

print("Repository:", REPO)
print("Dataset:", DATA_ROOT)
print("Category:", CATEGORY)
print("Image size:", IMAGE_SIZE)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

## 2. PCA implementation

Let X contain one flattened normal training image per row. We subtract the training mean from every row:

Xc = X - mean(X)

In the usual feature-space formulation, PCA uses the covariance matrix Xcᵀ Xc. At 256 × 256 pixels this matrix would be very large. We instead eigendecompose the smaller sample-space matrix:

G = Xc Xcᵀ

If u is an eigenvector of G with eigenvalue λ, the corresponding unit-length principal direction in pixel space is:

v = Xcᵀ u / sqrt(λ)

This gives the same principal directions without building the large pixel covariance matrix. We retain enough components to explain the selected fraction of training variance.

For a new image x, projection and reconstruction are:

z = (x - mean) Vᵀ

x̂ = z V + mean

The squared difference between the original and reconstructed pixels forms the anomaly map. Large reconstruction error means the image contains patterns that are not well represented by the normal training subspace.

In [ ]:
class GlobalPCA:
    def __init__(self, retained_variance=0.95):
        if not 0.0 < retained_variance <= 1.0:
            raise ValueError("retained_variance must be between 0 and 1.")
        self.retained_variance = retained_variance
        self.mean = None
        self.components = None
        self.explained_variance = None
        self.image_shape = None

    def fit(self, training_images):
        images = np.asarray(training_images, dtype=np.float64)

        if images.ndim != 4:
            raise ValueError("Images must have shape (N, C, H, W).")
        if images.shape[0] < 2:
            raise ValueError("At least two normal training images are required.")
        if not np.isfinite(images).all():
            raise ValueError("Training images contain invalid values.")

        self.image_shape = images.shape[1:]
        flat_images = images.reshape(images.shape[0], -1)
        self.mean = flat_images.mean(axis=0)
        centered_images = flat_images - self.mean

        sample_matrix = centered_images @ centered_images.T
        eigenvalues, eigenvectors = np.linalg.eigh(sample_matrix)

        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = np.maximum(eigenvalues[order], 0.0)
        eigenvectors = eigenvectors[:, order]

        valid = eigenvalues > eigenvalues[0] * 1e-10
        eigenvalues = eigenvalues[valid]
        eigenvectors = eigenvectors[:, valid]

        if len(eigenvalues) == 0:
            raise ValueError("The training images do not contain measurable variance.")

        principal_components = (eigenvectors.T @ centered_images) / np.sqrt(eigenvalues[:, None])
        variance_ratios = eigenvalues / eigenvalues.sum()
        cumulative_variance = np.cumsum(variance_ratios)
        component_count = np.searchsorted(cumulative_variance, self.retained_variance) + 1

        self.components = principal_components[:component_count]
        self.explained_variance = variance_ratios[:component_count]
        return self

    def reconstruct(self, images):
        if self.components is None:
            raise RuntimeError("Fit the model before reconstruction.")

        images = np.asarray(images, dtype=np.float64)
        if images.shape[1:] != self.image_shape:
            raise ValueError("The input image dimensions do not match the training images.")

        flat_images = images.reshape(images.shape[0], -1)
        centered_images = flat_images - self.mean
        reduced_images = centered_images @ self.components.T
        restored_images = reduced_images @ self.components + self.mean

        return restored_images.reshape(images.shape).astype(np.float32)

    def anomaly_map(self, images):
        images = np.asarray(images, dtype=np.float32)
        reconstructed_images = self.reconstruct(images)
        squared_error = (images - reconstructed_images) ** 2
        return squared_error.mean(axis=1)

    def save(self, file_path):
        if self.components is None:
            raise RuntimeError("Fit the model before saving.")

        np.savez_compressed(
            file_path,
            mean=self.mean,
            components=self.components,
            explained_variance=self.explained_variance,
            image_shape=np.asarray(self.image_shape)
        )

## 3. Check the implementation on synthetic images

The checks verify output dimensions, finite reconstruction errors and near-perfect reconstruction when all useful components are retained.

In [ ]:
random_generator = np.random.default_rng(SEED)
toy_images = random_generator.normal(size=(8, 1, 16, 16)).astype(np.float32)

toy_model = GlobalPCA(retained_variance=1.0).fit(toy_images)
toy_reconstruction = toy_model.reconstruct(toy_images)
toy_anomaly_map = toy_model.anomaly_map(toy_images)

assert toy_reconstruction.shape == toy_images.shape
assert toy_anomaly_map.shape == (8, 16, 16)
assert np.isfinite(toy_anomaly_map).all()
assert toy_anomaly_map.min() >= 0
assert np.mean((toy_images - toy_reconstruction) ** 2) < 1e-8

print("Global PCA synthetic checks passed.")

## 4. Load the MVTec splits

The dataset loader creates a validation subset from normal training images. Confirm that both train and validation contain only good samples before fitting.

In [ ]:
train_dataset = build_dataset(CONFIG, "pca", "train", CATEGORY)
validation_dataset = build_dataset(CONFIG, "pca", "val", CATEGORY)
test_dataset = build_dataset(CONFIG, "pca", "test", CATEGORY)

training_images = stack_images(train_dataset).cpu().numpy()

assert all(sample[2] == "good" for sample in train_dataset.samples)
assert all(sample[2] == "good" for sample in validation_dataset.samples)

print("Training images:", training_images.shape)
print("Validation images:", len(validation_dataset))
print("Test images:", len(test_dataset))

## 5. Fit global PCA

The default model retains 95% of the variance measured on normal training images. This value can be changed, but any tuning must be done using validation data rather than test performance.

In [ ]:
RETAINED_VARIANCE = 0.95

global_model = GlobalPCA(retained_variance=RETAINED_VARIANCE)
global_model.fit(training_images)

print("Number of principal components:", len(global_model.components))
print("Retained training variance:", global_model.explained_variance.sum())

## 6. Visualize a normal validation image

The error map is the squared pixel-wise difference between the original image and its PCA reconstruction.

In [ ]:
example = validation_dataset[0]
example_image = example["image"].unsqueeze(0).cpu().numpy()

original_image = example_image[0, 0]
reconstructed_image = global_model.reconstruct(example_image)[0, 0]
error_map = global_model.anomaly_map(example_image)[0]

figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(original_image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(reconstructed_image, cmap="gray")
axes[1].set_title("PCA reconstruction")
axes[2].imshow(error_map, cmap="inferno")
axes[2].set_title("Anomaly map")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7. Evaluation

The image threshold is set from the 95th percentile of normal validation image scores. The threshold is then fixed before evaluating the official test split.

AUROC and average precision measure ranking quality. Precision, recall and F1 use the validation-derived image threshold. Pixel metrics evaluate defect localization. AUPRO measures average overlap with connected defect regions up to 30% pixel false-positive rate.

In [ ]:
def collect_outputs(dataset, model):
    image_labels = []
    image_scores = []
    masks = []
    anomaly_maps = []

    for index in range(len(dataset)):
        item = dataset[index]
        image = item["image"].unsqueeze(0).cpu().numpy()
        anomaly_map = model.anomaly_map(image)[0]
        mask = item["mask"].squeeze(0).cpu().numpy().astype(bool)

        image_labels.append(int(item["label"]))
        image_scores.append(float(anomaly_map.max()))
        masks.append(mask)
        anomaly_maps.append(anomaly_map)

    return (
        np.asarray(image_labels, dtype=np.uint8),
        np.asarray(image_scores, dtype=np.float64),
        np.stack(masks),
        np.stack(anomaly_maps)
    )


def compute_aupro(masks, anomaly_maps, max_fpr=0.30, number_of_thresholds=128):
    masks = np.asarray(masks, dtype=bool)
    anomaly_maps = np.asarray(anomaly_maps, dtype=np.float64)

    regions = []
    for mask in masks:
        region_labels, number_of_regions = connected_components(mask)
        for region_number in range(1, number_of_regions + 1):
            region = region_labels == region_number
            if region.any():
                regions.append(region)

    negative_pixels = int((~masks).sum())
    if not regions or negative_pixels == 0:
        return float("nan")

    thresholds = np.unique(
        np.quantile(anomaly_maps, np.linspace(0.0, 1.0, number_of_thresholds))
    )[::-1]

    false_positive_rates = [0.0]
    region_overlaps = [0.0]

    for threshold in thresholds:
        prediction = anomaly_maps >= threshold
        false_positive_rate = np.logical_and(prediction, ~masks).sum() / negative_pixels

        if false_positive_rate > max_fpr:
            break

        overlaps = []
        for region in regions:
            overlaps.append(np.logical_and(prediction, region).sum() / region.sum())

        false_positive_rates.append(float(false_positive_rate))
        region_overlaps.append(float(np.mean(overlaps)))

    x_values = np.asarray(false_positive_rates)
    y_values = np.asarray(region_overlaps)
    order = np.argsort(x_values)
    x_values = x_values[order]
    y_values = y_values[order]

    unique_x = np.unique(x_values)
    unique_y = np.asarray([y_values[x_values == value].max() for value in unique_x])

    inside = unique_x < max_fpr
    x_values = unique_x[inside]
    y_values = unique_y[inside]

    if len(x_values) == 0 or x_values[0] != 0.0:
        x_values = np.insert(x_values, 0, 0.0)
        y_values = np.insert(y_values, 0, 0.0)

    end_value = float(np.interp(max_fpr, unique_x, unique_y))
    x_values = np.append(x_values, max_fpr)
    y_values = np.append(y_values, end_value)

    return float(np.clip(np.trapz(y_values, x_values) / max_fpr, 0.0, 1.0))


def evaluate_model(model, validation_dataset, test_dataset, model_name):
    validation_labels, validation_scores, _, _ = collect_outputs(validation_dataset, model)
    if np.any(validation_labels != 0):
        raise ValueError("Validation data must contain normal images only.")

    image_threshold = float(np.quantile(validation_scores, 0.95, method="higher"))

    test_labels, test_scores, test_masks, test_maps = collect_outputs(test_dataset, model)
    if np.unique(test_labels).size != 2:
        raise ValueError("The test split must contain both normal and anomalous images.")

    image_predictions = (test_scores >= image_threshold).astype(np.uint8)
    flat_masks = test_masks.reshape(-1).astype(np.uint8)
    flat_scores = test_maps.reshape(-1)

    results = {
        "category": CATEGORY,
        "method": model_name,
        "image_auroc": float(roc_auc_score(test_labels, test_scores)),
        "image_ap": float(average_precision_score(test_labels, test_scores)),
        "image_precision": float(precision_score(test_labels, image_predictions, zero_division=0)),
        "image_recall": float(recall_score(test_labels, image_predictions, zero_division=0)),
        "image_f1": float(f1_score(test_labels, image_predictions, zero_division=0)),
        "pixel_auroc": float(roc_auc_score(flat_masks, flat_scores)),
        "pixel_ap": float(average_precision_score(flat_masks, flat_scores)),
        "aupro_30pct_fpr": compute_aupro(test_masks, test_maps),
        "validation_threshold": image_threshold
    }

    return pd.DataFrame([results]), test_maps, test_masks

global_metrics, global_test_maps, global_test_masks = evaluate_model(
    global_model,
    validation_dataset,
    test_dataset,
    "Global PCA"
)

display(global_metrics.round(4))

## 8. Save the model and results

The saved model contains the training mean and retained principal components. The CSV contains the test metrics for this category.

In [ ]:
OUTPUT_DIR = Path("experiments/pca")
RESULTS_DIR = Path("results/tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / f"{CATEGORY}_global_pca.npz"
metrics_path = RESULTS_DIR / f"{CATEGORY}_global_pca_metrics.csv"

global_model.save(model_path)
global_metrics.to_csv(metrics_path, index=False)

print("Model saved to:", model_path)
print("Metrics saved to:", metrics_path)

## 9. Interpretation and limitations

- Global PCA learns a single subspace for whole images. It can capture broad appearance and alignment patterns, but may miss small local defects.
- Reconstruction error is an anomaly score, not a probability.
- The 95% retained-variance setting is a baseline. It should be compared with other settings using validation data only.
- Results in this notebook cover the selected category only. Repeat the same protocol for the other MVTec categories before making dataset-wide claims.
- The PCA detector is trained only on normal images; defect labels and masks are used only during final evaluation.